# JRA Pedigree Graph Study: Reproduction

This notebook displays the complete released focused results and recomputes **all 72 seed-level AP and Hit@3 point scores** from frozen predictions and keyed labels. It does not retrain models or rerun the 2,000-draw bootstrap. The test cohort was previously examined, so the comparisons remain exploratory.

Code and paper: https://github.com/CharisBerlian/jra-pedigree-graph-study

Data: https://www.kaggle.com/datasets/charischerw/jra-pedigree-graph-study-2010-2025 (version 1).

The four arms contain no odds or popularity. Historical market results are a separate analysis. Research data/results are CC BY 4.0; original source records are acknowledged to Netkeiba. Notebook code is MIT.


In [ ]:
from pathlib import Path
import csv, gzip, io, json, os, zipfile
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

DATA = Path(os.environ.get("JRA_RELEASE_DATA", "/kaggle/input"))
def read_released(path):
    matches = [p for p in DATA.rglob(Path(path).name) if p.as_posix().endswith(path)]
    if len(matches) == 1:
        return matches[0].read_bytes()
    for archive in DATA.rglob("*.zip"):
        with zipfile.ZipFile(archive) as z:
            if path in z.namelist():
                return z.read(path)
    if path.endswith(".gz"):
        # Kaggle expands GZIP members during dataset ingestion.
        return gzip.compress(read_released(path[:-3]), compresslevel=1)
    raise FileNotFoundError(path)

arm = pd.read_csv(io.BytesIO(read_released("companion/focused/test/arm_metrics.csv")))
reference = pd.read_csv(io.BytesIO(read_released("companion/focused/test/seed_metrics.csv")))
assert len(arm) == 24 and len(reference) == 72
display(arm)


## Recompute point metrics

Labels are joined by race ID, horse ID and horse number. Every prediction file must match the complete roster for its surface. AP groups equal scores at a common threshold. Hit@3 uses descending score, then ascending horse number and horse identifier. Races with shared top-three finish positions retain all positive labels, while three horses are selected per race.


In [ ]:
key = ["race_id", "horse_id_netkeiba", "horse_number"]
dtypes = {"race_id": str, "horse_id_netkeiba": str, "horse_number": int}
outcomes = pd.read_csv(io.BytesIO(gzip.decompress(read_released("outcomes/targeted.csv.gz"))), dtype=dtypes)
assert len(outcomes) == 89137 and not outcomes.duplicated(key).any()
assert outcomes["label"].isin([0, 1]).all()
assert outcomes.groupby("race_id").size().equals(outcomes.groupby("race_id")["field_size"].first())
recomputed = []
for row in reference.itertuples(index=False):
    name = f"predictions/{row.family}/{row.surface}/{row.arm}/seed{row.seed}/test_predictions.csv.gz"
    pred = pd.read_csv(io.BytesIO(gzip.decompress(read_released(name))), dtype=dtypes)
    truth = outcomes[outcomes["surface"].str.lower() == row.surface]
    assert len(pred) == len(truth) and not pred.duplicated(key).any()
    joined = truth.merge(pred[key + ["prediction"]], on=key, how="outer", validate="one_to_one", indicator=True)
    assert joined["_merge"].eq("both").all()
    assert np.isfinite(joined["prediction"]).all() and joined["prediction"].between(0, 1).all()
    ap = float(average_precision_score(joined["label"], joined["prediction"]))
    top = joined.sort_values(["race_id", "prediction", "horse_number", "horse_id_netkeiba"], ascending=[True, False, True, True]).groupby("race_id", sort=False).head(3)
    hit = float((top.groupby("race_id")["label"].sum() / 3).mean())
    assert abs(ap - row.average_precision) < 1e-12, (row, ap)
    assert abs(hit - row.hit_at_3) < 1e-12, (row, hit)
    recomputed.append({"surface": row.surface, "family": row.family, "arm": row.arm, "seed": row.seed, "average_precision": ap, "hit_at_3": hit})
checked = pd.DataFrame(recomputed)
display(checked)
print("PASS: all 72 seed-level AP and Hit@3 point scores match the released references.")


## Interpretation and reproduction limits

The ancestor-rule graph pipeline did not consistently improve AP over history or the parent-performance comparator. Turf primary AP contrasts were negative; dirt primary intervals included zero. Point-score reproduction does not establish full training replication, restore an untouched test, or reproduce the bootstrap intervals. Use the full packet's `verify.py` and its recorded environment for the frozen interval replay.
